In [12]:
import argparse
import json
import time
import traceback
from pathlib import Path

import numpy as np
from scipy.linalg import eigh

from qiskit.quantum_info import SparsePauliOp
from qiskit.circuit.library import efficient_su2, real_amplitudes, n_local
from qiskit.primitives import StatevectorEstimator
from qiskit_algorithms import VQE
from qiskit_algorithms.utils import algorithm_globals
from qiskit_algorithms.optimizers import (
    SPSA, COBYLA, NFT, QNSPSA, ADAM, BOBYQA
)
from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_aer import AerSimulator
from qiskit_aer.noise import NoiseModel
from qiskit_algorithms.state_fidelities import ComputeUncompute
from qiskit.primitives import StatevectorSampler
from qiskit_aer.primitives import EstimatorV2 as AerEstimator

In [13]:
QiskitRuntimeService.save_account(
    channel = "ibm_quantum_platform",
    token = 'Oj1-j4grN4RkHyeAnW1na3BU1rjqnbeFm9mTxHf_M8Tk',
    overwrite = True)

In [14]:
seed = 64
algorithm_globals.random_seed = seed
np.random.seed(seed)

# Hamiltonian Definition
def hamil(n_qubits, Jz = 1.0):
    terms = []
    for i in range(n_qubits - 1):
        for pauli_char in ["X", "Y"]:
            p = ["I"] * n_qubits
            p[i] = pauli_char
            p[i+1] = pauli_char
            terms.append(("".join(p), 1.0)) # join(p) converts list of characters to string, e.g. ['X', 'X', 'I'] -> 'XXI'

        pz = ["I"] * n_qubits
        pz[i] = "Z"
        pz[i+1] = "Z"
        terms.append(("".join(pz), Jz))

    return SparsePauliOp.from_list(terms)

# Antiferromagnetic Structure Factor S(π) - Order Parameter
def s_pi_op(n_qubits): 
    terms = []
    for i in range(n_qubits):
        for j in range(i+1, n_qubits):
            p = ["I"] * n_qubits
            p[i] = "Z"
            p[j] = "Z"
            coeff = ((-1)**(i-j))  / n_qubits
            terms.append(("".join(p), coeff))

    return SparsePauliOp.from_list(terms)

# Finds exact groundstate classically
def exact_gs(hamil: SparsePauliOp):
    h_mat = np.array(hamil.to_matrix())
    eigenvalues, eigenvectors = eigh(h_mat)
    return float(eigenvalues[0]), eigenvectors[:, 0] # ground state energy and vector


# Now building the ansatz
def build_ansatz(name, n_qubits, reps, entanglement = "linear"):
    if name == "efficient_su2":
        return efficient_su2(n_qubits, reps=reps, entanglement=entanglement).decompose()
    elif name == "real_amplitudes":
        return real_amplitudes(n_qubits, reps=reps, entanglement=entanglement).decompose()
    elif name == "n_local":
        return n_local(n_qubits, reps=reps, rotation_blocks=["ry", "rz"], entanglement_blocks = "cz").decompose()

# Build Optimizer
# Tried QNSPSA, ADAM, BOBYQA but they were either extremely poor performing or didn't work
def build_optimizer(name, maxiter):
    opts = {
        "SPSA": lambda: SPSA(maxiter=maxiter),
        "COBYLA": lambda: COBYLA(maxiter=maxiter),
        "NFT": lambda: NFT(maxiter=maxiter),

    }
    return opts[name]()

# Coupling Map

coupling_map = [[0, 1], [1, 2], [2, 3], [3, 4], [4, 5], [5, 6], [6, 7]]

# Build noise model

service = QiskitRuntimeService()
real_backend = service.backend("ibm_fez")
noise_model = NoiseModel.from_backend(real_backend)

# Build the noisy backend simulator from the real
noisy_backend = AerSimulator.from_backend(real_backend)

aer_sim = AerEstimator() 
aer_sim.options.simulator = {
    "method": "density_matrix",        # required for realistic noise simulation
    "coupling_map": coupling_map,
    "noise_model": noise_model,
}
aer_sim.options.default_precision = 1e-2

# Sweep

def run_config(ansatz_name, reps, optimizer_name, n_qubits, jz_vals, maxiter):
    ansatz = build_ansatz(ansatz_name, n_qubits, reps)
    circuit_depth = ansatz.depth()
    n_params = ansatz.num_parameters
    estimator = StatevectorEstimator()
    s_pi_mat = np.array(s_pi_op(n_qubits).to_matrix())
    optimizer = build_optimizer(optimizer_name, maxiter)

    all_counts, all_values = [], []

    def callback(eval_count, parameters, mean, std):
        if eval_count == 1:
            all_counts.append([])
            all_values.append([])
        all_counts[-1].append(eval_count)
        all_values[-1].append(mean)

    vqe = VQE(aer_sim, ansatz=ansatz, optimizer=optimizer, callback=callback)

    J_list, E_vqe, E_exact, Spi_vqe, Spi_exact = [], [], [], [], []
    opt_params = None
    wall_times = []

    for jz in jz_vals:
        H = hamil(n_qubits, Jz=jz)

        if opt_params is not None:
            vqe.initial_point = opt_params

        t0 = time.time()
        result = vqe.compute_minimum_eigenvalue(H)
        elapsed = time.time() - t0

        opt_params = result.optimal_point

        prim_result = estimator.run(
            [(ansatz, s_pi_op(n_qubits), opt_params)]
        ).result()
        spi_vqe = float(np.real(prim_result[0].data.evs))

        e_exact, psi0 = exact_gs(H)
        spi_exact = float(np.real(psi0.conj() @ s_pi_mat @ psi0))

        J_list.append(float(jz))
        E_vqe.append(float(np.real(result.eigenvalue)))
        E_exact.append(e_exact)
        Spi_vqe.append(spi_vqe)
        Spi_exact.append(spi_exact)
        wall_times.append(elapsed)

    energy_errors = [v - e for v, e in zip(E_vqe, E_exact)]

    return{
         # Config
        "ansatz":           ansatz_name,
        "reps":             reps,
        "optimizer":        optimizer_name,
        "n_qubits":         n_qubits,
        "n_params":         n_params,
        "circuit_depth":    circuit_depth,
        # Per-Jz arrays
        "jz_values":        J_list,
        "E_vqe":            E_vqe,
        "E_exact":          E_exact,
        "energy_errors":    energy_errors,
        "Spi_vqe":          Spi_vqe,
        "Spi_exact":        Spi_exact,
        "wall_times":       wall_times,
        # Summary metrics
        "mean_abs_error":   float(np.mean(np.abs(energy_errors))),
        "max_abs_error":    float(np.max(np.abs(energy_errors))),
        "total_time_s":     float(np.sum(wall_times)),
        # Convergence (list-of-lists, one per Jz)
        "conv_counts":      all_counts,
        "conv_values":      all_values,
    }

# ── Config — edit these directly instead of using command line args ────────────
n_qubits  = 8
jz_start  = -3.0
jz_end    =  3.0
jz_step   =  0.25
maxiter   =  300
output    = "sweep_results.json"

# ── Run ────────────────────────────────────────────────────────────────────────
jz_values = list(np.arange(jz_start, jz_end + jz_step / 2, jz_step)) 

ansatz_names    = ["efficient_su2", "real_amplitudes", "n_local"]
reps_options    = [1, 2]
optimizer_names = ["SPSA", "COBYLA", "NFT"]

configs = [
    (ans, rep, opt)
    for ans in ansatz_names
    for rep in reps_options
    for opt in optimizer_names
]

total = len(configs)
print(f"VQE Sweep: {total} configs x {len(jz_values)} Jz values each")
print(f"n_qubits={n_qubits}, maxiter={maxiter}, jz_step={jz_step}\n")

all_results = []
failed = []

for i, (ans, rep, opt) in enumerate(configs):
    label = f"[{i+1}/{total}] {ans} | reps={rep} | {opt}"
    print(f"{label} ...", end=" ", flush=True)
    t0 = time.time()

    try:
        result = run_config(ans, rep, opt, n_qubits, jz_values, maxiter)
        all_results.append(result)
        print(f"mean|ΔE|={result['mean_abs_error']:.4f}, depth={result['circuit_depth']}, t={time.time()-t0:.0f}s")
    except Exception as e:
        print(f"FAILED: {e}")
        failed.append({"config": label, "error": str(e), "traceback": traceback.format_exc()})

# Save
with open(output, "w") as f:
    json.dump({"metadata": {"n_qubits": n_qubits, "jz_step": jz_step,
                             "maxiter": maxiter, "n_configs": len(all_results),
                             "n_failed": len(failed)},
               "results": all_results, "failed": failed}, f, indent=2)

print(f"\nDone. {len(all_results)}/{total} completed. Saved to {output}")

# Quick ranking
ranked = sorted(all_results, key=lambda r: r["mean_abs_error"])
print(f"\n{'Ansatz':<16} {'Reps':<6} {'Optimizer':<10} {'Mean|ΔE|':<12} {'Depth'} ")
print("-" * 55)
for r in ranked[:5]:
    print(f"{r['ansatz']:<16} {r['reps']:<6} {r['optimizer']:<10} {r['mean_abs_error']:<12.5f} {r['circuit_depth']}")

qiskit_runtime_service.__init__:WARNING:2026-03-04 09:42:07,972: Instance was not set at service instantiation. Free and trial plan instances will be prioritized. Based on the following filters: (tags: None, region: us-east, eu-de), and available plans: (open), the available account instances are: open-instance. If you need a specific instance set it explicitly either by using a saved account with a saved default instance or passing it in directly to QiskitRuntimeService().
qiskit_runtime_service.backends:WARNING:2026-03-04 09:42:07,975: Using instance: open-instance, plan: open


VQE Sweep: 18 configs x 25 Jz values each
n_qubits=8, maxiter=300, jz_step=0.25

[1/18] efficient_su2 | reps=1 | SPSA ... mean|ΔE|=3.3338, depth=11, t=31s
[2/18] efficient_su2 | reps=1 | COBYLA ... mean|ΔE|=1.8430, depth=11, t=27s
[3/18] efficient_su2 | reps=1 | NFT ... mean|ΔE|=1.7973, depth=11, t=49s
[4/18] efficient_su2 | reps=2 | SPSA ... mean|ΔE|=3.9924, depth=15, t=36s
[5/18] efficient_su2 | reps=2 | COBYLA ... mean|ΔE|=1.5307, depth=15, t=34s
[6/18] efficient_su2 | reps=2 | NFT ... mean|ΔE|=0.8685, depth=15, t=56s
[7/18] real_amplitudes | reps=1 | SPSA ... mean|ΔE|=2.9299, depth=9, t=26s
[8/18] real_amplitudes | reps=1 | COBYLA ... mean|ΔE|=1.9762, depth=9, t=17s
[9/18] real_amplitudes | reps=1 | NFT ... mean|ΔE|=0.5897, depth=9, t=44s
[10/18] real_amplitudes | reps=2 | SPSA ... mean|ΔE|=4.1458, depth=12, t=32s
[11/18] real_amplitudes | reps=2 | COBYLA ... mean|ΔE|=2.2087, depth=12, t=23s
[12/18] real_amplitudes | reps=2 | NFT ... mean|ΔE|=0.5043, depth=12, t=44s
[13/18] n_local